In [1]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': True,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8],
  'custom1': [9, 11]},
 'method': 'welch',
 'stepSize': 0.3,
 'windowLength': 3}

In [3]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [5]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [6]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
25/04/21 21:33:39 WARN Utils: Your hostname, Mac-mini.local resolves to a loopback address: 127.0.0.1; using 192.168.1.172 instead (on interface en1)
25/04/21 21:33:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/21 21:33:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/21 21:33:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


New Spark session created successfully


In [7]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': True, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8], 'custom1': [9, 11]}, 'method': 'welch', 'stepSize': 0.3, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler

In [8]:
import pandas as pd

# alz_df_pandas = pd.read_pickle("alz_df_apr10_1355.pkl")
# cntrl_df_pandas = pd.read_pickle("cntrl_df_apr10_1355.pkl")


# alz_df_pandas = pd.read_pickle("features_alz_extra_features_Apr15_1033.pkl")
# cntrl_df_pandas = pd.read_pickle("features_cntrl_extra_features_Apr15_1033.pkl")

alz_df_pandas = pd.read_pickle("features_alz_extra_features_Apr19_2141.pkl")
cntrl_df_pandas = pd.read_pickle("features_cntrl_extra_features_Apr19_2141.pkl")


In [9]:
%%time
alz_df_spark = spark.createDataFrame(alz_df_pandas)
cntrl_df_spark = spark.createDataFrame(cntrl_df_pandas)

CPU times: user 6min 53s, sys: 14.9 s, total: 7min 8s
Wall time: 7min 16s


In [10]:
#just renaming things now that we understand the types and where things are coming from
alz_df = alz_df_spark
cntrl_df = cntrl_df_spark

In [11]:
alz_df.show()

25/04/21 21:41:10 WARN TaskSetManager: Stage 0 contains a task of very large size (69323 KiB). The maximum recommended task size is 1000 KiB.
25/04/21 21:41:15 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 0 (TID 0): Attempting to kill Python Worker
                                                                                

+---------+-------+---------+--------+-----------+--------------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|        FeatureValue|table_type|
+---------+-------+---------+--------+-----------+--------------------+----------+
|  sub-008|   ep-0|      Fp1|   Alpha|      Power|7.020328193902969E-4|      band|
|  sub-008|   ep-0|      Fp1|    Beta|      Power|3.399499109946191...|      band|
|  sub-008|   ep-0|      Fp1|   Delta|      Power| 0.08723169565200806|      band|
|  sub-008|   ep-0|      Fp1|   Theta|      Power|0.001139140920713544|      band|
|  sub-008|   ep-0|      Fp1| custom1|      Power|4.279543645679950...|      band|
|  sub-008|   ep-0|      Fp1|    NULL|TotalEnergy| 0.34805238246917725| electrode|
|  sub-008|   ep-0|      Fp1|    NULL| TotalPower| 0.01123595517128706| electrode|
|  sub-008|   ep-0|      Fp2|   Alpha|      Power|0.001468957751058042|      band|
|  sub-008|   ep-0|      Fp2|    Beta|      Power|5.043480778113008E-4|      band|
|  s

# Raw Data Visualization

In [35]:
alz_df.count()

14161135

# Start of data processing

In [12]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [13]:
# union everything
full_df = alz_df.unionByName(cntrl_df)

In [14]:
# Split based on feature type
from pyspark.sql.functions import col

band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")


In [15]:
from pyspark.sql.functions import concat_ws

# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName")).repartition(16).persist()

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName")).repartition(16).persist()

# Epoch-level: just FeatureName
epoch_df = epoch_df.withColumn("pivot", col("FeatureName")).repartition(16).persist()


In [16]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

25/04/21 21:41:15 WARN TaskSetManager: Stage 1 contains a task of very large size (69323 KiB). The maximum recommended task size is 1000 KiB.
25/04/21 21:41:26 WARN TaskSetManager: Stage 4 contains a task of very large size (57564 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [17]:
from functools import reduce

# full_df = reduce(
#     lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
#     [band_pivot, channel_pivot, epoch_pivot]
# ).fillna(0.0)


full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [band_pivot, channel_pivot, epoch_pivot]# band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)
full_df.repartition(16).persist()


25/04/21 21:41:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: double, C3_Beta_Power: double, C3_Delta_Power: double, C3_Theta_Power: double, C3_custom1_Power: double, C4_Alpha_Power: double, C4_Beta_Power: double, C4_Delta_Power: double, C4_Theta_Power: double, C4_custom1_Power: double, Cz_Alpha_Power: double, Cz_Beta_Power: double, Cz_Delta_Power: double, Cz_Theta_Power: double, Cz_custom1_Power: double, F3_Alpha_Power: double, F3_Beta_Power: double, F3_Delta_Power: double, F3_Theta_Power: double, F3_custom1_Power: double, F4_Alpha_Power: double, F4_Beta_Power: double, F4_Delta_Power: double, F4_Theta_Power: double, F4_custom1_Power: double, F7_Alpha_Power: double, F7_Beta_Power: double, F7_Delta_Power: double, F7_Theta_Power: double, F7_custom1_Power: double, F8_Alpha_Power: double, F8_Beta_Power: double, F8_Delta_Power: double, F8_Theta_Power: double, F8_custom1_Power: double, Fp1_Alpha_Power: double, Fp1_Beta_Power: double, Fp1_Delta_Power: double, Fp1_Theta_Power: doub

In [18]:
type(full_df)

pyspark.sql.dataframe.DataFrame

In [19]:
full_df.repartition(16).persist()


25/04/21 21:41:48 WARN CacheManager: Asked to cache already cached data.


DataFrame[SubjectID: string, EpochID: string, label: int, C3_Alpha_Power: double, C3_Beta_Power: double, C3_Delta_Power: double, C3_Theta_Power: double, C3_custom1_Power: double, C4_Alpha_Power: double, C4_Beta_Power: double, C4_Delta_Power: double, C4_Theta_Power: double, C4_custom1_Power: double, Cz_Alpha_Power: double, Cz_Beta_Power: double, Cz_Delta_Power: double, Cz_Theta_Power: double, Cz_custom1_Power: double, F3_Alpha_Power: double, F3_Beta_Power: double, F3_Delta_Power: double, F3_Theta_Power: double, F3_custom1_Power: double, F4_Alpha_Power: double, F4_Beta_Power: double, F4_Delta_Power: double, F4_Theta_Power: double, F4_custom1_Power: double, F7_Alpha_Power: double, F7_Beta_Power: double, F7_Delta_Power: double, F7_Theta_Power: double, F7_custom1_Power: double, F8_Alpha_Power: double, F8_Beta_Power: double, F8_Delta_Power: double, F8_Theta_Power: double, F8_custom1_Power: double, Fp1_Alpha_Power: double, Fp1_Beta_Power: double, Fp1_Delta_Power: double, Fp1_Theta_Power: doub

In [20]:
print("w")

w


In [21]:
NUM_TEST_SUBJECTS_PER_GROUP = 2

# Get test subject IDs from full_df (which has .label)
alz_test_subjects = (
    full_df.filter("label == 1")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

cntrl_test_subjects = (
    full_df.filter("label == 0")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

test_subjects = alz_test_subjects + cntrl_test_subjects #this will be the firs 2 subjects of each group for reproduceablility

In [22]:
print("w")

w


In [23]:
# from pyspark.sql.functions import rand

# NUM_TEST_SUBJECTS_PER_GROUP = 2
# SEED = 42

# # Alzheimer's test subjects (label == 1)
# alz_test_subjects = (
#     full_df.filter("label == 1")
#     .select("SubjectID")
#     .distinct()
#     .orderBy(rand(SEED))  # Randomize with seed
#     .limit(NUM_TEST_SUBJECTS_PER_GROUP)
#     .rdd.flatMap(lambda row: row)
#     .collect()
# )

# # Control test subjects (label == 0)
# cntrl_test_subjects = (
#     full_df.filter("label == 0")
#     .select("SubjectID")
#     .distinct()
#     .orderBy(rand(SEED + 1))  # Different seed for different shuffle
#     .limit(NUM_TEST_SUBJECTS_PER_GROUP)
#     .rdd.flatMap(lambda row: row)
#     .collect()
# )

# # Combine test subjects
# test_subjects = alz_test_subjects + cntrl_test_subjects


In [24]:
print(f"alz_test_subjects {alz_test_subjects}")
print(f"cntrl_test_subjects {cntrl_test_subjects}")

alz_test_subjects ['sub-001', 'sub-002']
cntrl_test_subjects ['sub-037', 'sub-038']


In [25]:
# Split into test and train sets
test_df = full_df.filter(col("SubjectID").isin(test_subjects))
train_df = full_df.filter(~col("SubjectID").isin(test_subjects))


In [26]:
print("got here") 

got here


# DO T-TEST HERE !!

In [38]:
train_df.columns

['SubjectID',
 'EpochID',
 'label',
 'C3_Alpha_Power',
 'C3_Beta_Power',
 'C3_Delta_Power',
 'C3_Theta_Power',
 'C3_custom1_Power',
 'C4_Alpha_Power',
 'C4_Beta_Power',
 'C4_Delta_Power',
 'C4_Theta_Power',
 'C4_custom1_Power',
 'Cz_Alpha_Power',
 'Cz_Beta_Power',
 'Cz_Delta_Power',
 'Cz_Theta_Power',
 'Cz_custom1_Power',
 'F3_Alpha_Power',
 'F3_Beta_Power',
 'F3_Delta_Power',
 'F3_Theta_Power',
 'F3_custom1_Power',
 'F4_Alpha_Power',
 'F4_Beta_Power',
 'F4_Delta_Power',
 'F4_Theta_Power',
 'F4_custom1_Power',
 'F7_Alpha_Power',
 'F7_Beta_Power',
 'F7_Delta_Power',
 'F7_Theta_Power',
 'F7_custom1_Power',
 'F8_Alpha_Power',
 'F8_Beta_Power',
 'F8_Delta_Power',
 'F8_Theta_Power',
 'F8_custom1_Power',
 'Fp1_Alpha_Power',
 'Fp1_Beta_Power',
 'Fp1_Delta_Power',
 'Fp1_Theta_Power',
 'Fp1_custom1_Power',
 'Fp2_Alpha_Power',
 'Fp2_Beta_Power',
 'Fp2_Delta_Power',
 'Fp2_Theta_Power',
 'Fp2_custom1_Power',
 'Fz_Alpha_Power',
 'Fz_Beta_Power',
 'Fz_Delta_Power',
 'Fz_Theta_Power',
 'Fz_custom1_Po

In [39]:
# import dimensionality_reduction
# import importlib
# importlib.reload(dimensionality_reduction)
# from dimensionality_reduction import min_max_normalize, normalize_by_column_per_subject_wide
# feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]


# # train_norm_df, test_norm_df = min_max_normalize(train_df, test_df, feature_cols)
# train_norm_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_norm_df = normalize_by_column_per_subject_wide(test_df, feature_cols)

In [40]:
train_df.head(1)

[Row(SubjectID='sub-003', EpochID='ep-1003', label=1, C3_Alpha_Power=0.0012282916577532887, C3_Beta_Power=0.00024898923584260046, C3_Delta_Power=0.08426367491483688, C3_Theta_Power=0.0037428848445415497, C3_custom1_Power=0.00141051912214607, C4_Alpha_Power=0.0023519701790064573, C4_Beta_Power=0.0002878349623642862, C4_Delta_Power=0.08370231091976166, C4_Theta_Power=0.002958987606689334, C4_custom1_Power=0.003850673558190465, Cz_Alpha_Power=0.0013206840958446264, Cz_Beta_Power=0.0001966899144463241, Cz_Delta_Power=0.08362051099538803, Cz_Theta_Power=0.004475411027669907, Cz_custom1_Power=0.001396479899995029, F3_Alpha_Power=0.0036768284626305103, F3_Beta_Power=0.00024767022114247084, F3_Delta_Power=0.07738445699214935, F3_Theta_Power=0.007606237195432186, F3_custom1_Power=0.002669800305739045, F4_Alpha_Power=0.0031283418647944927, F4_Beta_Power=0.00026330543914809823, F4_Delta_Power=0.07639987766742706, F4_Theta_Power=0.00898689404129982, F4_custom1_Power=0.003181564388796687, F7_Alpha_

In [29]:
# train_df.schema

In [36]:
# import dimensionality_reduction
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide
feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]

# train_norm_df, test_norm_df = train_df, test_df

# train_norm_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_norm_df = normalize_by_column_per_subject_wide(test_df, feature_cols)

# RUNNING T-TEST - using regression to do the test like named here 
# https://stackoverflow.com/questions/58851008/how-to-perform-student-t-test-in-pyspark

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

results = []
for feat in feature_cols:
    # Step 1: Assemble the single feature into featuresCol
    assembler = VectorAssembler(inputCols=["label"], outputCol="features")
    assembled = assembler.transform(train_df.select("label", feat).dropna())

    # Step 2: Fit regression model: feature ~ label
    lr = LinearRegression(featuresCol="features", labelCol=feat, regParam=0)
    model = lr.fit(assembled)

    # Step 3: Get t-stat and p-value for 'label'
    summary = model.summary
    t_stat = summary.tValues[1]  # index 1 corresponds to label coefficient
    p_val = summary.pValues[1]

    # Save results
    results.append((feat, t_stat, p_val))




results_df = pd.DataFrame(results, columns=["Feature", "T_statistic", "P_value"])
results_df.sort_values("P_value", inplace=True)  # sort by significance


# add benferroni or FDR correction 
# from statsmodels.stats.multitest import multipletests

# # Apply FDR correction
# rejected, pvals_corrected, _, _ = multipletests(results_df["P_value"], alpha=0.05, method="fdr_bh")
# results_df["FDR_corrected"] = pvals_corrected
# results_df["Significant"] = rejected


train_norm_df.repartition(16).persist()
test_norm_df.repartition(16).persist()
print("finished T-test and have results")

25/04/21 21:46:55 WARN Instrumentation: [2fd0e005] regParam is zero, which might cause numerical instability and overfitting.
25/04/21 21:47:17 WARN Instrumentation: [56559a5e] regParam is zero, which might cause numerical instability and overfitting.
25/04/21 21:47:39 WARN Instrumentation: [836d5e1e] regParam is zero, which might cause numerical instability and overfitting.
25/04/21 21:47:58 WARN Instrumentation: [36256b34] regParam is zero, which might cause numerical instability and overfitting.
25/04/21 21:48:20 WARN Instrumentation: [bb028985] regParam is zero, which might cause numerical instability and overfitting.
25/04/21 21:48:41 WARN Instrumentation: [cf9f2386] regParam is zero, which might cause numerical instability and overfitting.
25/04/21 21:49:03 WARN Instrumentation: [bdb553b6] regParam is zero, which might cause numerical instability and overfitting.
25/04/21 21:49:25 WARN Instrumentation: [ba7ba2c6] regParam is zero, which might cause numerical instability and overf

Py4JJavaError: An error occurred while calling o5618.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 7 in stage 5452.0 failed 1 times, most recent failure: Lost task 7.0 in stage 5452.0 (TID 15482) (192.168.1.172 executor driver): java.io.IOException: No space left on device
	at java.base/sun.nio.ch.FileDispatcherImpl.write0(Native Method)
	at java.base/sun.nio.ch.FileDispatcherImpl.write(FileDispatcherImpl.java:62)
	at java.base/sun.nio.ch.IOUtil.writeFromNativeBuffer(IOUtil.java:113)
	at java.base/sun.nio.ch.IOUtil.write(IOUtil.java:58)
	at java.base/sun.nio.ch.FileChannelImpl.write(FileChannelImpl.java:280)
	at java.base/sun.nio.ch.FileChannelImpl.transferToTrustedChannel(FileChannelImpl.java:589)
	at java.base/sun.nio.ch.FileChannelImpl.transferTo(FileChannelImpl.java:682)
	at org.apache.spark.util.Utils$.copyFileStreamNIO(Utils.scala:346)
	at org.apache.spark.util.Utils.copyFileStreamNIO(Utils.scala)
	at org.apache.spark.shuffle.sort.BypassMergeSortS

In [ ]:
# train_norm_df.head(1)
results_df


In [ ]:
import os
os.system('say "t-test is done!"')

25/04/21 22:07:01 WARN TaskSetManager: Lost task 5.0 in stage 5452.0 (TID 15480) (192.168.1.172 executor driver): TaskKilled (Stage cancelled: Job aborted due to stage failure: Task 7 in stage 5452.0 failed 1 times, most recent failure: Lost task 7.0 in stage 5452.0 (TID 15482) (192.168.1.172 executor driver): java.io.IOException: No space left on device
	at java.base/sun.nio.ch.FileDispatcherImpl.write0(Native Method)
	at java.base/sun.nio.ch.FileDispatcherImpl.write(FileDispatcherImpl.java:62)
	at java.base/sun.nio.ch.IOUtil.writeFromNativeBuffer(IOUtil.java:113)
	at java.base/sun.nio.ch.IOUtil.write(IOUtil.java:58)
	at java.base/sun.nio.ch.FileChannelImpl.write(FileChannelImpl.java:280)
	at java.base/sun.nio.ch.FileChannelImpl.transferToTrustedChannel(FileChannelImpl.java:589)
	at java.base/sun.nio.ch.FileChannelImpl.transferTo(FileChannelImpl.java:682)
	at org.apache.spark.util.Utils$.copyFileStreamNIO(Utils.scala:346)
	at org.apache.spark.util.Utils.copyFileStreamNIO(Utils.scala)


In [56]:
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
from dimensionality_reduction import apply_pca_model

train_df = apply_pca_model(train_norm_df, pca_input_cols, pca_model, k_val)
test_df = apply_pca_model(test_norm_df, pca_input_cols, pca_model, k_val)


In [57]:
train_df

DataFrame[SubjectID: string, EpochID: string, label: int, features: vector]

In [59]:
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
    
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide

# train_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_df = normalize_by_column_per_subject_wide(test_df, feature_cols)


In [60]:
print("got here")

got here


# ML time

In [106]:
train_df.head(1)

25/04/21 13:46:53 WARN DAGScheduler: Broadcasting large task binary with size 1390.5 KiB
25/04/21 13:46:53 WARN DAGScheduler: Broadcasting large task binary with size 1899.6 KiB


[Row(SubjectID='sub-008', EpochID='ep-1016', label=1, features=DenseVector([6.7014, 1.8475, -0.687, 1.9046, -1.3617, 1.3174, 0.3283, 0.3859, -1.2671, -0.2785, -0.1831, 0.5881, 0.0421, -0.8095, -0.3075, 0.2311, -0.118, -0.0023, -0.1569, 0.4231, 0.3328, -0.0829, 2.4159, 0.0065, 0.3049, 0.3517, 0.2699, -0.0844, 0.479, -0.238, 0.4183, 0.1636, 0.0319, 0.1977]))]

In [107]:
train_df.columns

['SubjectID', 'EpochID', 'label', 'features']

In [111]:
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
importlib.reload(dimensionality_reduction)
from dimensionality_reduction import min_max_normalize_post_pca_by_subject

train_df = min_max_normalize_post_pca_by_subject(train_df)
test_df = min_max_normalize_post_pca_by_subject(test_df)


25/04/21 13:51:18 WARN DAGScheduler: Broadcasting large task binary with size 1390.5 KiB
25/04/21 13:51:18 WARN DAGScheduler: Broadcasting large task binary with size 1897.7 KiB
25/04/21 13:51:50 WARN DAGScheduler: Broadcasting large task binary with size 1390.3 KiB
25/04/21 13:51:51 WARN DAGScheduler: Broadcasting large task binary with size 1897.3 KiB


In [112]:
train_df.head(1)

25/04/21 13:52:46 WARN DAGScheduler: Broadcasting large task binary with size 1390.5 KiB
25/04/21 13:52:48 WARN DAGScheduler: Broadcasting large task binary with size 1390.5 KiB
25/04/21 13:52:48 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/04/21 13:52:49 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB


[Row(SubjectID='sub-008', EpochID='ep-1016', label=1, features=DenseVector([0.929, 0.1546, 0.5244, 0.4944, 0.2445, 0.4597, 0.6894, 0.5245, 0.4003, 0.5227, 0.3875, 0.206, 0.4933, 0.8361, 0.5382, 0.8665, 0.4331, 0.464, 0.507, 0.5211, 0.6491, 0.2816, 0.7972, 0.4203, 0.5493, 0.5576, 0.3549, 0.5629, 0.6093, 0.4541, 0.7693, 0.5723, 0.6143, 0.6384]))]

In [113]:
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()
# spark.stop()

25/04/21 13:53:31 WARN DAGScheduler: Broadcasting large task binary with size 1390.5 KiB
25/04/21 13:53:33 WARN DAGScheduler: Broadcasting large task binary with size 1390.5 KiB
25/04/21 13:53:34 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/04/21 13:53:35 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
25/04/21 13:54:17 WARN DAGScheduler: Broadcasting large task binary with size 1390.3 KiB
25/04/21 13:54:18 WARN DAGScheduler: Broadcasting large task binary with size 1390.3 KiB
25/04/21 13:54:18 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
25/04/21 13:54:19 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB


In [122]:
import numpy as np

# Convert Spark DenseVectors to regular 2D numpy arrays
X_train = np.array(train_pd["features"].tolist())
y_train = train_pd["label"].values

X_test = np.array(test_pd["features"].tolist())
y_test = test_pd["label"].values


In [123]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


X_train shape: (168652, 34)
y_train shape: (168652,)


In [124]:
y_train

array([1, 1, 1, ..., 0, 0, 0], dtype=int32)

In [125]:
X_train

array([[0.92902373, 0.15457149, 0.5243658 , ..., 0.57228269, 0.61430329,
        0.63840763],
       [0.92490272, 0.1558201 , 0.53242566, ..., 0.5742086 , 0.6099872 ,
        0.61824714],
       [0.89457419, 0.0898189 , 0.61841167, ..., 0.55256905, 0.60846818,
        0.61361105],
       ...,
       [0.55238146, 0.27357219, 0.66509698, ..., 0.67251047, 0.50911422,
        0.62650332],
       [0.55975266, 0.32487147, 0.72139604, ..., 0.51227069, 0.33555397,
        1.        ],
       [0.80343293, 0.31695814, 0.45614341, ..., 0.57956534, 0.49776886,
        0.58772661]])

In [126]:
# How can we do standard scaler per subject ! !!!  ! ! ! !  !  !! ! !  ! 

In [127]:
print("work")

work


In [128]:
from sklearn.preprocessing import StandardScaler


X_train_scaled = X_train

X_test_scaled = X_test

# making sure min-maxed ! also might change results a little 

# scaler = StandardScaler()

# X_train_scaled = scaler.fit_transform(X_train)

# X_test_scaled = scaler.transform(X_test)

In [129]:
%%time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np
import time

# # Step 1: Scale once
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# Step 2: Define hyperparameter grid
k_values = [1, 2, 3, 5, 7, 9, 11]
weights_list = ['uniform', 'distance']
metrics = ['euclidean', 'manhattan']
p_values = [1, 2]  # Only used if metric is 'minkowski'

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]
print("start")

# Step 4: Manual hyperparameter search
for k in k_values:
    for weight in weights_list:
        for metric in metrics:
            for p in p_values:

                if metric != 'minkowski' and p != 2:
                    continue  # p is irrelevant unless using 'minkowski'

                label = f"KNN k={k}, weight={weight}, metric={metric}, p={p}"
                model = KNeighborsClassifier(
                    n_neighbors=k,
                    weights=weight,
                    metric=metric if metric != 'minkowski' else 'minkowski',
                    p=p
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=15, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        # Step 7: Evaluate on the held-out test set
                        #!! shouldn't we be testing oin the model that is trained on all the folds 1 by 1 (so multiple epochs) ?
                        model.fit(X_train_scaled, y_train)
                        y_test_pred = model.predict(X_test_scaled)
                        test_acc = accuracy_score(y_test, y_test_pred)
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_test, y_test_pred, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 8: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Test: {test_acc:.4f}")


start

=== Cross-Validation: KNN k=1, weight=uniform, metric=euclidean, p=2 ===
Mean Accuracy: 0.9996
Std Deviation: 0.0002
All Fold Scores: [0.9999 0.9995 0.9995 0.9997 0.9998 0.9998 0.9996 0.9996 0.9994 0.9996
 0.9996 0.9993 0.9996 0.9997 0.9995]

=== Best Fold Summary: KNN k=1, weight=uniform, metric=euclidean, p=2 ===
Train Accuracy: 1.0000
Validation Accuracy: 0.9998
              precision    recall  f1-score   support

     Control       1.00      1.00      1.00      5041
 Alzheimer's       1.00      1.00      1.00      6203

    accuracy                           1.00     11244
   macro avg       1.00      1.00      1.00     11244
weighted avg       1.00      1.00      1.00     11244

Test Accuracy: 0.4822
              precision    recall  f1-score   support

     Control       0.52      0.59      0.55      5552
 Alzheimer's       0.42      0.36      0.38      4624

    accuracy                           0.48     10176
   macro avg       0.47      0.47      0.47     10176
weig

KeyboardInterrupt: 

In [130]:
%%time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once (IMPORTANT: transform X_test with same scaler)
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: Hyperparameters
layer_configs = [(256, 128, 64), (128, 64, 16)]
activations = ['relu']
alphas = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 5e-1, 1.0]
early_stopping_options = [True, False]
max_iter = 30000

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force hyperparameter loop
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}, max_iter={max_iter}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=max_iter,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Evaluate on best validation fold
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        break

                # Step 7: Evaluate on held-out test set
                model.fit(X_train_scaled, y_train)
                y_test_pred = model.predict(X_test_scaled)
                test_acc = accuracy_score(y_test, y_test_pred)
                print(f"\n=== Final Test Set Evaluation: {label} ===")
                print(f"Test Accuracy: {test_acc:.4f}")
                print(classification_report(y_test, y_test_pred, target_names=target_names))

                results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 8: Print final summary
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Test: {test_acc:.4f}")



=== Cross-Validation: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=True, max_iter=30000 ===
Mean Accuracy: 0.9913
Std Deviation: 0.0006
All Fold Scores: [0.9921 0.9918 0.9912 0.9902 0.9913]

=== Best Fold Summary: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=True, max_iter=30000 ===
Train Accuracy: 0.9991
Validation Accuracy: 0.9921
              precision    recall  f1-score   support

     Control       0.99      0.99      0.99     15123
 Alzheimer's       0.99      0.99      0.99     18608

    accuracy                           0.99     33731
   macro avg       0.99      0.99      0.99     33731
weighted avg       0.99      0.99      0.99     33731


=== Final Test Set Evaluation: MLP (256, 128, 64), act=relu, alpha=1e-06, early_stop=True, max_iter=30000 ===
Test Accuracy: 0.5503
              precision    recall  f1-score   support

     Control       0.60      0.53      0.56      5552
 Alzheimer's       0.50      0.57      0.54      4624

    accuracy          

/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/admin/neuro-venv/lib/python3.9/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


KeyboardInterrupt: 

In [131]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

# Define models
models = {
    # "KNN-tuned": KNeighborsClassifier(
    #     n_neighbors=3,
    #     weights='distance',
    #     metric='euclidean',
    #     p=1
    # ),
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=8,
        max_features=None,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42
    ),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "BaggedSVM": make_pipeline(
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),
    "SVM": make_pipeline(
        SVC(kernel='linear', probability=True)
    )
}

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            val_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Validation Accuracy: {val_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break

    # Final evaluation on the held-out test set
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    print(f"\n=== Test Set Evaluation: {name} ===")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(classification_report(y_test, y_test_pred, target_names=["Control", "Alzheimer's"]))



=== Cross-Validation: DecisionTree ===
Mean Accuracy: 0.8532
Standard Deviation: 0.0057
All Fold Scores: [0.8587 0.8495 0.8602 0.8552 0.8479 0.8416 0.8492 0.8636 0.854  0.8495
 0.8513 0.8532 0.8605 0.8562 0.8478]

=== Best Fold Summary: DecisionTree ===
Train Accuracy: 0.8638
Validation Accuracy: 0.8588
              precision    recall  f1-score   support

     Control       0.85      0.84      0.84      5040
 Alzheimer's       0.87      0.88      0.87      6203

    accuracy                           0.86     11243
   macro avg       0.86      0.86      0.86     11243
weighted avg       0.86      0.86      0.86     11243


=== Test Set Evaluation: DecisionTree ===
Test Accuracy: 0.6730
              precision    recall  f1-score   support

     Control       0.67      0.78      0.72      5552
 Alzheimer's       0.68      0.54      0.60      4624

    accuracy                           0.67     10176
   macro avg       0.67      0.66      0.66     10176
weighted avg       0.67      0

/Users/admin/neuro-venv/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Mean Accuracy: 0.7600
Standard Deviation: 0.0139
All Fold Scores: [0.7524 0.7582 0.7828 0.7649 0.743  0.7436 0.7562 0.7812 0.7624 0.746
 0.7534 0.7616 0.7893 0.7572 0.7478]

=== Best Fold Summary: BaggedSVM ===
Train Accuracy: 0.7630
Validation Accuracy: 0.7561
              precision    recall  f1-score   support

     Control       0.75      0.68      0.71      5041
 Alzheimer's       0.76      0.82      0.79      6202

    accuracy                           0.76     11243
   macro avg       0.76      0.75      0.75     11243
weighted avg       0.76      0.76      0.75     11243


=== Test Set Evaluation: BaggedSVM ===
Test Accuracy: 0.7130
              precision    recall  f1-score   support

     Control       0.69      0.86      0.77      5552
 Alzheimer's       0.77      0.53      0.63      4624

    accuracy                           0.71     10176
   macro avg       0.73      0.70      0.70     10176
weighted avg       0.72      0.71      0.70     10176


=== Cross-Validation:

KeyboardInterrupt: 

In [134]:
%%time
from sklearn.ensemble import BaggingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: Scale data for SVMs
scaler = StandardScaler()
X_train_svm = scaler.fit_transform(X_train)
X_test_svm = scaler.transform(X_test)

# Step 2: Define hyperparameter grids
C_values = [0.01, 0.1, 1, 10]
n_estimators_list = [5, 10, 20]
max_samples_list = [0.1, 0.5, 1.0]
bootstrap_options = [False, True]

# Step 3: Track results
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Hyperparameter tuning
for C in C_values:
    for n_est in n_estimators_list:
        for max_samp in max_samples_list:
            for bootstrap in bootstrap_options:
                
                label = f"BaggedSVM C={C}, est={n_est}, max_samples={max_samp}, bootstrap={bootstrap}"
                model = make_pipeline(
                    BaggingClassifier(
                        estimator=SVC(C=C, kernel='linear', probability=False),
                        n_estimators=n_est,
                        max_samples=max_samp,
                        bootstrap=bootstrap,
                        n_jobs=3,
                        random_state=42
                    )
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                scores = cross_val_score(model, X_train_svm, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Best fold deep dive
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, val_idx) in enumerate(skf.split(X_train_svm, y_train)):
                    if i == best_fold_index:
                        X_tr, X_val = X_train_svm[train_idx], X_train_svm[val_idx]
                        y_tr, y_val = y_train[train_idx], y_train[val_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_val = model.predict(X_val)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_val, y_pred_val)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_val, y_pred_val, target_names=target_names))

                        break

                # Final test set evaluation
                model.fit(X_train_svm, y_train)
                y_test_pred = model.predict(X_test_svm)
                test_acc = accuracy_score(y_test, y_test_pred)

                print(f"\n=== Test Set Evaluation: {label} ===")
                print(f"Test Accuracy: {test_acc:.4f}")
                print(classification_report(y_test, y_test_pred, target_names=target_names))

                results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 5: Print top models
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<90} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")



=== Cross-Validation: BaggedSVM C=0.01, est=5, max_samples=0.1, bootstrap=False ===
Mean Accuracy: 0.7603
Std Deviation: 0.0069
All Fold Scores: [0.7661 0.7495 0.7672 0.7547 0.764 ]

=== Best Fold Summary: BaggedSVM C=0.01, est=5, max_samples=0.1, bootstrap=False ===
Train Accuracy: 0.7618
Validation Accuracy: 0.7632
              precision    recall  f1-score   support

     Control       0.77      0.67      0.72     15122
 Alzheimer's       0.76      0.84      0.80     18608

    accuracy                           0.76     33730
   macro avg       0.76      0.75      0.76     33730
weighted avg       0.76      0.76      0.76     33730


=== Test Set Evaluation: BaggedSVM C=0.01, est=5, max_samples=0.1, bootstrap=False ===
Test Accuracy: 0.7080
              precision    recall  f1-score   support

     Control       0.69      0.85      0.76      5552
 Alzheimer's       0.75      0.54      0.63      4624

    accuracy                           0.71     10176
   macro avg       0.72  

KeyboardInterrupt: 

In [ ]:
%%time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: No need to scale for trees
X_train_tree = X_train
X_test_tree = X_test

# Step 2: More complex hyperparameter grid
max_depths = [10, 15, 20, None]  # None = fully grow the tree
min_samples_splits = [2, 3, 5]   # Smaller split thresholds
min_samples_leafs = [1, 2]       # Smaller leaves allowed

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Manual hyperparameter search
for depth in max_depths:
    for min_split in min_samples_splits:
        for min_leaf in min_samples_leafs:

            label = f"DecisionTree max_depth={depth}, min_split={min_split}, min_leaf={min_leaf}"
            model = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_split=min_split,
                min_samples_leaf=min_leaf,
                max_features=None,  # use all features
                ccp_alpha=0.0,      # disable pruning
                random_state=42
            )

            print(f"\n=== Cross-Validation: {label} ===")
            start = time.time()

            # Step 5: Cross-validation scores
            scores = cross_val_score(model, X_train_tree, y_train, cv=5, scoring='accuracy', n_jobs=3)
            mean_acc, std_acc = scores.mean(), scores.std()
            print(f"Mean Accuracy: {mean_acc:.4f}")
            print(f"Std Deviation: {std_acc:.4f}")
            print(f"All Fold Scores: {np.round(scores, 4)}")

            # Step 6: Best fold evaluation
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            best_fold_index = np.argmax(scores)
            for i, (train_idx, test_idx) in enumerate(skf.split(X_train_tree, y_train)):
                if i == best_fold_index:
                    X_tr, X_te = X_train_tree[train_idx], X_train_tree[test_idx]
                    y_tr, y_te = y_train[train_idx], y_train[test_idx]

                    model.fit(X_tr, y_tr)
                    y_pred_test = model.predict(X_te)
                    y_pred_train = model.predict(X_tr)

                    train_acc = accuracy_score(y_tr, y_pred_train)
                    test_acc = accuracy_score(y_te, y_pred_test)

                    print(f"\n=== Best Fold Summary: {label} ===")
                    print(f"Train Accuracy: {train_acc:.4f}")
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_te, y_pred_test, target_names=target_names))

                    results.append((label, mean_acc, std_acc, train_acc, test_acc))
                    break

            print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np


# Define models
models = { # NEED TO GO THORUGH AND MAKE SURE DEFAULTS ARE OK, SHOULD NOT HAVE HAD STANDARD SCALER FOR SVM'S
    # "KNN-tuned": KNeighborsClassifier(
    # n_neighbors=3,
    # weights='distance',
    # metric='euclidean',
    # p=1
    # ),
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=8,
        max_features=None,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42
    ),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
        # Faster ensemble of SVMs
    "BaggedSVM": make_pipeline(
        # StandardScaler(),
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),

    # Classic SVM (can be slow on big data)
    "SVM": make_pipeline(
        # StandardScaler(), 
        SVC(kernel='linear',
        probability=True)
    )
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break


# ML experiments and tuning

Notes and concerns:
 Are we folding correclty, it looks like we are retraining the best fold, ??? super weird
 Also seems like we need to do more iterations for SVM ... we can get some more out of it 

 Since hte neural nets that havve best aucuracy almost have 1000% in training, should try and do regurlization to try and 'even' out the accuracy between the two

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
import numpy as np
import time


import warnings
from sklearn.exceptions import ConvergenceWarning

# Suppress ConvergenceWarnings only
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Set up test space
kernels = ['linear', 'rbf', 'sigmoid']
Cs = [0.1, 1, 10]
gammas = ['scale', 0.01]  # for rbf/sigmoid
n_estimators = 10
max_iter = 10000

# Replace with your target class names
target_names = ["Control", "Alzheimer's"]

# Loop over kernel/C/gamma
for kernel in kernels:
    for C in Cs:
        if kernel == 'linear':
            label = f"BaggedSVM - linear, C={C}"
            base_model = SVC(kernel=kernel, C=C, probability=False, max_iter=max_iter)
            model = make_pipeline(
                BaggingClassifier(
                    estimator=base_model,
                    n_estimators=n_estimators,
                    max_samples=0.1,
                    n_jobs=3,
                    bootstrap=False,
                    random_state=42
                )
            )
            print(f"\n=== Cross-Validation: {label} ===")
            start = time.time()

            scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
            print(f"Mean Accuracy: {scores.mean():.4f}")
            print(f"Standard Deviation: {scores.std():.4f}")
            print(f"All Fold Scores: {np.round(scores, 4)}")

            best_fold_index = np.argmax(scores)
            skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
            for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
                if i == best_fold_index:
                    X_tr, X_te = X_train[train_index], X_train[test_index]
                    y_tr, y_te = y_train[train_index], y_train[test_index]

                    model.fit(X_tr, y_tr)
                    y_pred_test = model.predict(X_te)
                    y_pred_train = model.predict(X_tr)

                    test_acc = accuracy_score(y_te, y_pred_test)
                    train_acc = accuracy_score(y_tr, y_pred_train)

                    print(f"\n=== Best Fold Summary: {label} ===")
                    print(f"Train Accuracy: {train_acc:.4f}")
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_te, y_pred_test, target_names=target_names))
                    break

            print(f"⏱️ Duration: {time.time() - start:.1f} sec")

        elif kernel in ['rbf', 'sigmoid']:
            for gamma in gammas:
                label = f"BaggedSVM - {kernel}, C={C}, gamma={gamma}"
                base_model = SVC(kernel=kernel, C=C, gamma=gamma, probability=False, max_iter=max_iter)
                model = make_pipeline(
                    BaggingClassifier(
                        estimator=base_model,
                        n_estimators=n_estimators,
                        max_samples=0.1,
                        n_jobs=3,
                        bootstrap=False,
                        random_state=42
                    )
                )
                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
                print(f"Mean Accuracy: {scores.mean():.4f}")
                print(f"Standard Deviation: {scores.std():.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                best_fold_index = np.argmax(scores)
                skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
                for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train[train_index], X_train[test_index]
                        y_tr, y_te = y_train[train_index], y_train[test_index]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        test_acc = accuracy_score(y_te, y_pred_test)
                        train_acc = accuracy_score(y_tr, y_pred_train)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f} sec")


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once
scaler = MinMaxScaler(feature_range=(-1, 1))  # You can use (0, 1) if you prefer
X_train_scaled = scaler.fit_transform(X_train)

# Step 2: Define hyperparameter grids
layer_configs = [(100,), (128,), (128, 64), (256, 128, 64)]
activations = ['relu', 'tanh']
alphas = [1e-4, 1e-3, 1e-2]
early_stopping_options = [True, False]

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force loop over hyperparameter combinations
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=10000,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation accuracy scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Find best fold for detailed summary
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
%%time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

# Step 1: Scale features if needed (optional for GB, but useful with continuous variables)
X_train_boost = X_train_scaled  # assuming you already have this
X_test_boost = X_test_scaled

# Step 2: Define hyperparameter grid for more complex boosting
n_estimators_list = [100, 200]
learning_rates = [0.05, 0.1]
max_depths = [3, 5, 7]
min_samples_leafs = [1, 3]

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Manual hyperparameter search
for n_est in n_estimators_list:
    for lr in learning_rates:
        for depth in max_depths:
            for min_leaf in min_samples_leafs:

                label = f"GradBoost n={n_est}, lr={lr}, depth={depth}, min_leaf={min_leaf}"
                model = GradientBoostingClassifier(
                    n_estimators=n_est,
                    learning_rate=lr,
                    max_depth=depth,
                    min_samples_leaf=min_leaf,
                    random_state=42
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_boost, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_boost, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_boost[train_idx], X_train_boost[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 7: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

import os

# Define models
models = { # NEED TO GO THORUGH AND MAKE SURE DEFAULTS ARE OK, SHOULD NOT HAVE HAD STANDARD SCALER FOR SVM'S
    # "KNN": KNeighborsClassifier(),
    # "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    # "DecisionTree": DecisionTreeClassifier(
    #     max_depth=8,
    #     max_features=None,
    #     min_samples_leaf=10,
    #     min_samples_split=5,
    #     random_state=42
    # ),
    # "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42),
        # Faster ensemble of SVMs
    # "BaggedSVM": make_pipeline(
    #     StandardScaler(),
    #     BaggingClassifier(
    #         estimator=SVC(kernel='linear', probability=False),
    #         n_estimators=10,
    #         max_samples=0.1,
    #         n_jobs=3,
    #         bootstrap=False,
    #         random_state=42
    #     )
    # ),
    # Classic SVM (can be slow on big data)
    "BaggedSVM": make_pipeline(
        # StandardScaler(),
        BaggingClassifier(
            estimator=SVC(kernel='linear', probability=False),
            n_estimators=10,
            max_samples=0.1,
            n_jobs=3,
            bootstrap=False,
            random_state=42
        )
    ),
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break

os.system('say "SVM is done!"')

# ML Testing/tuning # best for nets, tanh 200 hidden layers

In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

# Define models
models = {
    # "KNN": KNeighborsClassifier(),
    # "SVM": make_pipeline(StandardScaler(), SVC(probability=True)),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(200,), activation='tanh', max_iter=10000, random_state=42)
    # "DecisionTree": DecisionTreeClassifier(
    #     max_depth=8,
    #     max_features=None,
    #     min_samples_leaf=10,
    #     min_samples_split=5,
    #     random_state=42
    # ),
    # "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break


In [ ]:
model = DecisionTreeClassifier(
    max_depth=8,
    max_features=None,
    min_samples_leaf=10,
    min_samples_split=5,
    random_state=42
)

print(f"\n=== Cross-Validation: Regularized DecisionTree ===")

# Cross-validation scores
scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)

print(f"Mean Accuracy: {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")
print(f"All Fold Scores: {np.round(scores, 4)}")

# Find the best-performing fold
best_fold_index = np.argmax(scores)

# Evaluate best fold manually
skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
    if i == best_fold_index:
        X_tr, X_te = X_train[train_index], X_train[test_index]
        y_tr, y_te = y_train[train_index], y_train[test_index]

        model.fit(X_tr, y_tr)

        y_pred_train = model.predict(X_tr)
        y_pred_test = model.predict(X_te)

        train_acc = accuracy_score(y_tr, y_pred_train)
        test_acc = accuracy_score(y_te, y_pred_test)

        print(f"\n=== Best Fold Summary: Regularized DecisionTree ===")
        print(f"Train Accuracy: {train_acc:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}")
        print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
        break


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    'max_depth': [4, 6, 8],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 'log2', None]
}

tree = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(tree, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)